In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Spoilage Alert Dataset
def generate_spoilage_data(n_batches=500):
    start_date = datetime(2025, 1, 1)
    dates = [start_date + timedelta(hours=i*6) for i in range(n_batches)]

    data = {
        "batch_id": [f"BATCH_{i}" for i in range(n_batches)],
        "date_time": dates,
        "temperature_C": np.random.normal(4, 2, n_batches),
        "humidity_%": np.random.normal(70, 10, n_batches),
        "transport_delay_hr": np.random.randint(0, 24, n_batches),
    }
    df = pd.DataFrame(data)
    # Spoilage probability based on conditions
    df["spoilage_probability"] = (
        0.05 + 0.02*(df["temperature_C"]-4) + 0.01*(df["humidity_%"]-70) + 0.03*(df["transport_delay_hr"]/12)
    ).clip(0,1)
    df["alert_flag"] = (df["spoilage_probability"] > 0.5).astype(int)
    return df

# Demand Forecasting Dataset
def generate_demand_data(n_days=365):
    start_date = datetime(2025, 1, 1)
    dates = [start_date + timedelta(days=i) for i in range(n_days)]
    ports = ["Lagos", "Calabar"]

    data = []
    for d in dates:
        season = "festive" if d.month in [12,1] else "normal"
        for port in ports:
            demand = np.random.poisson(500 if season=="normal" else 700)
            supply = demand + np.random.randint(-50, 50)
            price = round(np.random.uniform(800, 1200), 2)
            weather_index = np.random.uniform(0.5, 1.5)
            data.append([d, port, demand, supply, price, season, weather_index])

    df = pd.DataFrame(data, columns=["date","port","demand_kg","supply_kg","price_per_kg","season","weather_index"])
    return df

# Example usage
spoilage_df = generate_spoilage_data()
forecast_df = generate_demand_data()

print(spoilage_df.head())
print(forecast_df.head())


  batch_id           date_time  temperature_C  humidity_%  transport_delay_hr  \
0  BATCH_0 2025-01-01 00:00:00       6.365817   64.018450                   5   
1  BATCH_1 2025-01-01 06:00:00       0.864941   66.425481                  12   
2  BATCH_2 2025-01-01 12:00:00       7.575156   67.771117                   8   
3  BATCH_3 2025-01-01 18:00:00       3.507758   58.625917                   1   
4  BATCH_4 2025-01-02 00:00:00       6.858331   64.919197                  21   

   spoilage_probability  alert_flag  
0              0.050001           0  
1              0.000000           0  
2              0.119214           0  
3              0.000000           0  
4              0.108859           0  
        date     port  demand_kg  supply_kg  price_per_kg   season  \
0 2025-01-01    Lagos        697        711       1164.00  festive   
1 2025-01-01  Calabar        667        672       1088.36  festive   
2 2025-01-02    Lagos        673        712       1104.44  festive   
3 202

In [2]:
# Generate datasets
spoilage_df = generate_spoilage_data()
forecast_df = generate_demand_data()

# Export to CSV
spoilage_df.to_csv("spoilage_alert_dataset.csv", index=False)
forecast_df.to_csv("demand_forecast_dataset.csv", index=False)

print("Datasets exported successfully!")


Datasets exported successfully!


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler

# Step 1: Generate synthetic demand dataset
def generate_demand_data(n_days=365):
    start_date = datetime(2025, 1, 1)
    dates = [start_date + timedelta(days=i) for i in range(n_days)]
    ports = ["Lagos", "Calabar"]

    data = []
    for d in dates:
        season = "festive" if d.month in [12,1] else "normal"
        for port in ports:
            base_demand = 500 if season=="normal" else 700
            demand = np.random.poisson(base_demand)
            supply = demand + np.random.randint(-50, 50)
            price = round(np.random.uniform(800, 1200), 2)
            weather_index = np.random.uniform(0.5, 1.5)
            data.append([d, port, demand, supply, price, season, weather_index])

    df = pd.DataFrame(data, columns=["date","port","demand_kg","supply_kg","price_per_kg","season","weather_index"])
    return df

# Step 2: Preprocessing
def preprocess_data(df):
    # Handle missing values
    df = df.fillna(method="ffill")

    # Encode categorical season
    df["season_flag"] = df["season"].map({"normal":0, "festive":1})

    # Scale numerical features
    scaler = MinMaxScaler()
    df[["demand_scaled","supply_scaled","price_scaled","weather_scaled"]] = scaler.fit_transform(
        df[["demand_kg","supply_kg","price_per_kg","weather_index"]]
    )

    # Add lag features (previous demand values)
    df["demand_lag1"] = df["demand_scaled"].shift(1)
    df["demand_lag7"] = df["demand_scaled"].shift(7)

    # Add rolling average
    df["demand_ma7"] = df["demand_scaled"].rolling(window=7).mean()

    # Drop NA from lag/rolling
    df = df.dropna()

    return df

# Step 3: Train/Test Split
def train_test_split(df, test_ratio=0.2):
    split_idx = int(len(df)*(1-test_ratio))
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]
    return train_df, test_df

# Example usage
df = generate_demand_data()
df_processed = preprocess_data(df)
train_df, test_df = train_test_split(df_processed)

print(train_df.head())
print(test_df.head())


         date     port  demand_kg  supply_kg  price_per_kg   season  \
7  2025-01-04  Calabar        666        676       1036.15  festive   
8  2025-01-05    Lagos        737        690       1114.30  festive   
9  2025-01-05  Calabar        698        747        908.78  festive   
10 2025-01-06    Lagos        665        691       1000.06  festive   
11 2025-01-06  Calabar        689        725       1031.16  festive   

    weather_index  season_flag  demand_scaled  supply_scaled  price_scaled  \
7        1.216275            1       0.636872       0.652913      0.589720   
8        1.058379            1       0.835196       0.686893      0.785575   
9        0.537180            1       0.726257       0.825243      0.270513   
10       0.825209            1       0.634078       0.689320      0.499273   
11       0.704817            1       0.701117       0.771845      0.577214   

    weather_scaled  demand_lag1  demand_lag7  demand_ma7  
7         0.717610     0.826816     0.715084 

/tmp/ipython-input-190/3840157880.py:29: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")
